In [3]:
import requests
import time
import csv
import os


SUBREDDIT = "canada"       
TOPIC = "tariffs"          
USER_AGENT = "Mozilla/5.0 (compatible; MyScraper/1.0)"
NUM_ARTICLES = 100        


SCRIPT_DIR = os.path.dirname(os.path.abspath("__file__"))


OUTPUT_CSV = os.path.join(SCRIPT_DIR, "webscraped.csv")
OUTPUT_TXT = os.path.join(SCRIPT_DIR, "webscraped.txt")

def fetch_posts(subreddit, topic, user_agent, num_articles=100):

    base_url = f"https://www.reddit.com/r/{subreddit}/search.json"
    all_posts = []
    limit = 25
    count = 0
    after = None
    headers = {"User-Agent": user_agent}

    start_time = time.time()

    while count < num_articles:
        params = {"q": topic, "restrict_sr": 1, "limit": limit, "after": after}
        response = requests.get(base_url, headers=headers, params=params)
        
        if response.status_code != 200:
            print(f"Failed to retrieve search results. Status: {response.status_code}")
            break
        
        json_data = response.json()
        posts = json_data.get("data", {}).get("children", [])

        if not posts:
            print("No more posts found.")
            break

        for post in posts:
            post_data = post.get("data", {})
            title = post_data.get("title", "No Title")
            permalink = post_data.get("permalink", "")
            reddit_url = "https://www.reddit.com" + permalink
            external_url = post_data.get("url", "")

            all_posts.append({
                "title": title,
                "reddit_url": reddit_url,
                "external_url": external_url,
                "permalink": permalink
            })
            count += 1


            print(f"[{count}/{num_articles}] Retrieved: {title}")
            print(f"Reddit URL: {reddit_url}")
            print(f"External Link: {external_url}\n")

            if count >= num_articles:
                break
        
        after = json_data["data"].get("after", None)
        if after is None:
            print("Reached the end of available posts.")
            break

        time.sleep(1)

    total_time = time.time() - start_time
    print(f"Collected {len(all_posts)} posts in {total_time:.2f} seconds.")
    
    if len(all_posts) == 0:
        print("No posts were collected! Check if the subreddit or topic is correct.")
    
    return all_posts

def fetch_comments(permalink, user_agent, max_comments=50, retries=3):

    url = f"https://www.reddit.com{permalink}.json"
    headers = {"User-Agent": user_agent}
    
    for attempt in range(1, retries + 1):
        try:
            response = requests.get(url, headers=headers, timeout=10)
            if response.status_code == 429:
                print(f"Rate limited! Waiting 10 seconds before retrying (attempt {attempt}/{retries})...")
                time.sleep(10)
                continue
            
            if response.status_code != 200:
                print(f"Failed to fetch comments: {url} (status {response.status_code})")
                return ""

            data = response.json()
            if len(data) < 2:
                return ""

            comments_list = data[1]["data"]["children"]
            collected_comments = []
            count = 0

            for item in comments_list:
                if item.get("kind") == "t1":
                    comment_data = item.get("data", {})
                    body = comment_data.get("body", "")
                    collected_comments.append(body)
                    count += 1
                    if count >= max_comments:
                        break

            return "\n".join(collected_comments)

        except Exception as e:
            print(f"Error fetching comments (attempt {attempt}/{retries}): {e}")
            time.sleep(3)

    print(f"Completely failed to fetch comments for {url} after {retries} retries.")
    return ""

def scrape_reddit_comments(subreddit, topic, user_agent, num_articles=100, max_comments=50):

    posts = fetch_posts(subreddit, topic, user_agent, num_articles)
    results = []

    if not posts:
        print("No posts found. Skipping comment retrieval.")
        return []

    for i, p in enumerate(posts, start=1):
        permalink = p["permalink"]
        title = p["title"]
        reddit_url = p["reddit_url"]
        external_url = p["external_url"]

        print(f"Fetching comments for post [{i}/{len(posts)}]...")
        comments_text = fetch_comments(permalink, user_agent, max_comments)
        print(f"Collected {len(comments_text.split())} words from comments.\n")

        results.append({
            "title": title,
            "reddit_url": reddit_url,
            "external_url": external_url,
            "comments_text": comments_text
        })
        time.sleep(1)

    return results

def save_to_csv(articles, output_file):
    fieldnames = ["title", "reddit_url", "external_url", "comments_text"]
    
    if not articles:
        print("No data to save to CSV.")
        return

    with open(output_file, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for article in articles:
            writer.writerow(article)

    print(f"Saved {len(articles)} entries to {output_file}")

def save_to_txt(articles, output_file):
    """
    Saves scraped articles to a text file.
    """
    if not articles:
        print("No data to save to TXT.")
        return

    with open(output_file, "w", encoding="utf-8") as f:
        for i, article in enumerate(articles, start=1):
            f.write(f"Title: {article['title']}\n")
            f.write(f"Reddit URL: {article['reddit_url']}\n")
            f.write(f"External Link: {article['external_url']}\n")
            f.write(f"Comments:\n{article['comments_text']}\n")
            f.write("-" * 80 + "\n")

    print(f"📂 Saved {len(articles)} entries to {output_file}")


if __name__ == "__main__":
    scraped_data = scrape_reddit_comments(SUBREDDIT, TOPIC, USER_AGENT, NUM_ARTICLES, max_comments=50)

    if scraped_data:
        save_to_csv(scraped_data, OUTPUT_CSV)
        save_to_txt(scraped_data, OUTPUT_TXT)
        print("Scraping complete!")
    else:
        print("Scraping failed or no data was retrieved.")


[1/100] Retrieved: ‘Only Works as a State’: Trump Vows Not ‘To Bend’ On Tariffs Until Canada Is Absorbed Into The U.S.
Reddit URL: https://www.reddit.com/r/canada/comments/1jaib2x/only_works_as_a_state_trump_vows_not_to_bend_on/
External Link: https://www.mediaite.com/news/only-works-as-a-state-trump-vows-not-to-bend-on-tariffs-until-canada-is-absorbed-into-the-u-s/

[2/100] Retrieved: Statement by the Prime Minister on unjustified U.S. tariffs against Canada
Reddit URL: https://www.reddit.com/r/canada/comments/1j3059b/statement_by_the_prime_minister_on_unjustified_us/
External Link: https://www.pm.gc.ca/en/news/statements/2025/03/03/statement-prime-minister-unjustified-us-tariffs-against-canada

[3/100] Retrieved: Canada retaliating for Trump’s tariffs with 25 per cent tariffs on billions of U.S. goods
Reddit URL: https://www.reddit.com/r/canada/comments/1ifmyka/canada_retaliating_for_trumps_tariffs_with_25_per/
External Link: https://www.ctvnews.ca/politics/article/canada-retaliating